# Challenge: ลงมือแก้โค้ดเอง
ที่ผ่านมาส่วนใหญ่กด Run  รอบนี้**แก้โค้ดเอง** ให้ผ่านตัวตรวจอัตโนมัติ  มี 8 ข้อ จากง่ายไปยาก แต่ละข้อมี
- **โจทย์** และสิ่งที่ต้องแก้ (มองหาคำว่า `TODO`)
- **ตัวตรวจ** รันแล้วบอกทันทีว่า ✅ ผ่าน หรือ ❌ ยัง
- **คำใบ้** ถ้าติด  และ**เฉลย**ซ่อนไว้ท้าย notebook (เปิดดูได้แต่ลองเองก่อน)

ทำเป็นคู่ ใช้เวลา 60 นาที  ผ่าน 4 ข้อ = ใช้ทักษะพื้นฐานได้  ผ่าน 6 ข้อขึ้นไป = ต่อยอดงานจริงได้

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath("../scripts"))
import cv2
import numpy as np
from vision_camera import show, img_path, put_label

def load(name):
    return cv2.imread(img_path(name))

def check(title, got, expected):
    ok = (got == expected)
    print(("✅ ผ่าน  " if ok else "❌ ยัง   ") + title + f"   ได้ {got}  ต้องการ {expected}")
    return ok

def count_color(img, h_low, h_high, min_area=1000, s_min=50, v_min=50):
    """นับวัตถุในช่วงสี H ที่ใหญ่กว่า min_area  (ฟังก์ชันจาก Workshop 2)"""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([h_low, s_min, v_min]), np.array([h_high, 255, 255]))
    cs, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    return sum(cv2.contourArea(c) > min_area for c in cs), mask
print("พร้อม")

## ข้อ 1 (ง่าย): นับสีน้ำเงิน
ใน `shapes.png` โค้ดด้านล่างนับสีแดง (H 0–10)  **แก้ช่วง H ให้นับสีน้ำเงิน**  (คำใบ้: จากสไลด์ HSV น้ำเงินอยู่ราว H 95–125)

In [ ]:
img = load("shapes.png")
n_blue, mask = count_color(img, 0, 10)        # TODO: แก้ช่วง H ให้เป็นสีน้ำเงิน
check("ข้อ 1 นับสีน้ำเงิน (ชิ้นใหญ่กว่า 1000 px)", n_blue, 3)
show(img, mask, titles=["shapes.png", "mask ที่ได้"], size=4)

## ข้อ 2 (ง่าย): ถาดต้องไม่มีสีส้มปน
ถาด `line/tray_05.png` มีชิ้นแดงใหญ่ครบ 5 แต่มี**ชิ้นสีส้ม**ปนมา (H ราว 12–22)  เขียนให้ `verdict` เป็น `"NG"` เมื่อมีสีส้ม

In [ ]:
img = load("line/tray_05.png")
n_red, _ = count_color(img, 0, 10)
n_orange = 0                                   # TODO: นับชิ้นสีส้มด้วย count_color
verdict = "OK" if n_red == 5 else "NG"         # TODO: เพิ่มเงื่อนไข ถ้ามีสีส้ม = NG
check("ข้อ 2 ถาดมีสีส้ม", verdict, "NG")
img2 = load("line/tray_01.png")
n_red2, _ = count_color(img2, 0, 10); n_orange2 = 0   # TODO: นับสีส้มของถาดนี้ด้วย
verdict2 = "OK" if n_red2 == 5 else "NG"               # TODO: ใช้เงื่อนไขเดียวกัน
check("ข้อ 2 ถาดปกติยังต้อง OK", verdict2, "OK")

## ข้อ 3 (ง่าย): เศษชิ้นแตก = NG
ถาด `line/tray_03.png` มีชิ้นแดงใหญ่ 5 ชิ้น แต่มี**เศษแดงเล็ก** 2 ชิ้นปน (พื้นที่ 50–1000 พิกเซล)  ให้นับเศษแล้วตัดสิน NG

In [ ]:
img = load("line/tray_03.png")
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
mask = cv2.inRange(hsv, np.array([0, 50, 50]), np.array([10, 255, 255]))
cs, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
areas = [cv2.contourArea(c) for c in cs]
n_big = sum(a > 1000 for a in areas)
n_chip = 0                                     # TODO: นับ contour ที่พื้นที่อยู่ระหว่าง 50 ถึง 1000
verdict = "OK" if n_big == 5 else "NG"         # TODO: ถ้ามีเศษ ให้เป็น NG
check("ข้อ 3 จำนวนเศษ", n_chip, 2)
check("ข้อ 3 ผลตรวจ", verdict, "NG")

## ข้อ 4 (กลาง): ฉลากต้องมีเลข LOT
จาก notebook 06 ข้อ 2  ฉลากที่ดีต้องมีคำว่า `LOT` ตามด้วยตัวเลข 6 หลัก  เขียนฟังก์ชัน `has_lot(text)` ให้คืน True/False
(คำใบ้: ใช้ `re.search(r"LOT\s*\d{6}", text)`)

In [ ]:
import re
def has_lot(text):
    return False                               # TODO: ตรวจด้วย regular expression

check("ข้อ 4 ฉลากมี LOT", has_lot("PRODUCT A LOT 775758 EXP 2027-03-15"), True)
check("ข้อ 4 LOT ไม่ครบ 6 หลัก", has_lot("PRODUCT A LOT 7757 EXP 2027-03-15"), False)
check("ข้อ 4 ไม่มี LOT", has_lot("PRODUCT A EXP 2027-03-15"), False)

## ข้อ 5 (กลาง): เพิ่มโซนตรวจไฟสถานะไดรฟ์ของ Rack server
จาก notebook 08 ข้อ 1  ไดรฟ์แต่ละตัวมี**ไฟสถานะสีเขียว**ที่ด้านล่าง (แถบ y 235–243 ของแต่ละช่อง)  ภาพ `rack/rack_05_drive_led_off.png` มีไดรฟ์ตัวหนึ่งไฟดับ
เขียนลูปตรวจไฟของไดรฟ์ทั้ง 8 ตัว แล้วคืนหมายเลขไดรฟ์ที่ไฟดับ (เริ่มที่ 1)

In [ ]:
img = load("rack/rack_05_drive_led_off.png")
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
led_off = []
for i in range(8):
    x = 50 + i * 88
    roi = hsv[233:245, x + 8:x + 72]           # กรอบแถบไฟของไดรฟ์ตัวที่ i
    green_ratio = 0.0                          # TODO: สัดส่วนพิกเซลสีเขียว (H 40-80, S>80, V>80) ใน roi
    if green_ratio < 0.3:
        led_off.append(i + 1)
check("ข้อ 5 ไดรฟ์ที่ไฟดับ", led_off, [3])

## ข้อ 6 (กลาง): วัดขนาดเมื่อเปลี่ยนวัตถุสอบเทียบ
จาก notebook 06 ข้อ 4  ถ้าเหรียญในภาพเดิมมีขนาดจริง **25 มม.** (ไม่ใช่ 20)  ชิ้นงานจะกว้างกี่มิลลิเมตร  คำนวณ `px_per_mm` ใหม่แล้ววัด (ปัดเป็นทศนิยม 1 ตำแหน่ง)

In [ ]:
img = load("industrial/measure_01.png")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
_, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
cs, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cs = sorted(cs, key=lambda c: cv2.boundingRect(c)[0])
(cx, cy), r = cv2.minEnclosingCircle(cs[0])
COIN_MM = 20.0                                 # TODO: เหรียญจริง 25 มม.
px_per_mm = (2 * r) / COIN_MM
w, h = cv2.minAreaRect(cs[1])[1]
width_mm = round(max(w, h) / px_per_mm, 1)
check("ข้อ 6 ความกว้างชิ้นงาน (มม.)", width_mm, 31.9)

## ข้อ 7 (ยาก): ตรวจสิ่งแปลกปลอมเมื่อถาดเลื่อน
`industrial/tray_06_shifted_screw.png` คือถาดที่มีสกรู แต่**เลื่อนไปจากภาพอ้างอิง** 14 พิกเซลทางขวา 9 พิกเซลขึ้น  วิธีเทียบภาพอ้างอิงตรงๆ จะเห็น "ทุกชิ้นต่าง"
ให้**หาระยะเลื่อน**ด้วย `cv2.matchTemplate` (ใช้ส่วนกลางของภาพอ้างอิงเป็น template) แล้วเลื่อนภาพกลับก่อนเทียบ  ต้องเจอความต่าง**เพียง 1 จุด** (สกรู)

In [ ]:
ref = load("industrial/tray_reference.png")
img = load("industrial/tray_06_shifted_screw.png")

def align(img, ref):
    """คืนภาพ img ที่เลื่อนให้ตรงกับ ref"""
    g_ref = cv2.cvtColor(ref, cv2.COLOR_BGR2GRAY)
    g_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    template = g_ref[100:380, 150:490]                       # ส่วนกลางของภาพอ้างอิง
    dx, dy = 0, 0                                            # TODO: หา dx, dy ด้วย matchTemplate
    #   res = cv2.matchTemplate(g_img, template, cv2.TM_CCOEFF_NORMED)
    #   _, _, _, (x, y) = cv2.minMaxLoc(res)   -> ตำแหน่งที่ template ไปอยู่ในภาพใหม่
    #   dx = x - 150,  dy = y - 100
    M = np.float32([[1, 0, -dx], [0, 1, -dy]])
    return cv2.warpAffine(img, M, (img.shape[1], img.shape[0]), borderValue=(70, 72, 75)), (dx, dy)

aligned, shift = align(img, ref)
g1 = cv2.GaussianBlur(cv2.cvtColor(aligned, cv2.COLOR_BGR2GRAY), (5, 5), 0)
g0 = cv2.GaussianBlur(cv2.cvtColor(ref, cv2.COLOR_BGR2GRAY), (5, 5), 0)
_, m = cv2.threshold(cv2.absdiff(g1, g0), 20, 255, cv2.THRESH_BINARY)
m = cv2.dilate(m, np.ones((5, 5), np.uint8))
cs, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
n_diff = sum(cv2.contourArea(c) > 120 for c in cs)
check("ข้อ 7 ระยะเลื่อนที่หาได้ (dx, dy)", shift, (14, -9))
check("ข้อ 7 จำนวนความต่างหลังจัดตำแหน่ง", n_diff, 1)
show(m, titles=[f"ความต่างหลังเลื่อนกลับ  shift={shift}"], size=5)

## ข้อ 8 (ยาก): เพิ่มประเภทที่ 4 ให้ AI
จาก notebook 07  เพิ่มประเภท `"extra"` (ถาดที่มีชิ้นงานเกินมา 1 ชิ้นวางตรงกลาง) แล้วสอนโมเดลใหม่  ความแม่นต้อง ≥ 85%
สิ่งที่ต้องแก้: (1) ฟังก์ชัน `make_tray` วาดชิ้นเกินเมื่อ kind == "extra"  (2) `CLASSES` มี 4 ประเภท

In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

CLASSES = ["ok", "missing", "foreign"]         # TODO: เพิ่ม "extra"

def make_tray(kind, rng):
    img = np.full((240, 320, 3), (70, 72, 75), np.uint8)
    dx, dy = rng.integers(-10, 11, 2)
    centers = [(60 + dx, 60 + dy), (160 + dx, 60 + dy), (260 + dx, 60 + dy),
               (60 + dx, 170 + dy), (160 + dx, 170 + dy), (260 + dx, 170 + dy)]
    skip = rng.integers(0, 6) if kind == "missing" else -1
    for i, (x, y) in enumerate(centers):
        if i == skip:
            continue
        cv2.circle(img, (int(x), int(y)), 26, (190, 190, 195), -1)
        cv2.circle(img, (int(x), int(y)), 8, (80, 82, 85), -1)
    if kind == "foreign":
        fx, fy = int(rng.integers(30, 290)), int(rng.integers(30, 210))
        cv2.rectangle(img, (fx, fy), (fx + int(rng.integers(40, 70)), fy + 12), (30, 30, 35), -1)
    # TODO: if kind == "extra": วาดชิ้นงาน (วงกลมเทา r=26 + รู r=8) เพิ่มที่ (160+dx, 115+dy)
    light = rng.uniform(0.6, 1.3)
    return np.clip(img.astype(np.float32) * light + rng.normal(0, 3, img.shape), 0, 255).astype(np.uint8)

from vision_camera import make_hog
hog = make_hog()   # OpenCV 4 ใช้ HOG ของ OpenCV  OpenCV 5 ใช้ตัวที่เขียนเองแทน
def features(img):
    small = cv2.resize(img, (64, 48)); gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    thumb = cv2.resize(gray, (32, 24)).astype(np.float32)
    hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)
    col = cv2.calcHist([hsv], [0, 1], None, [8, 4], [0, 180, 0, 256]).flatten()
    return np.concatenate([hog.compute(gray).flatten(), (thumb / (thumb.mean() + 1e-6)).flatten(), 10 * col / (col.sum() + 1e-6)])

rng = np.random.default_rng(0)
X, y = [], []
for ci, kind in enumerate(CLASSES):
    for _ in range(100):
        X.append(features(make_tray(kind, rng))); y.append(ci)
X, y = np.array(X), np.array(y)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
model = make_pipeline(StandardScaler(), SVC(C=10)).fit(X_tr, y_tr)
acc = round((model.predict(X_te) == y_te).mean() * 100)
print("ประเภท:", CLASSES, " ความแม่น", acc, "%")
check("ข้อ 8 มี 4 ประเภท", len(CLASSES), 4)
check("ข้อ 8 ความแม่น >= 85%", acc >= 85, True)

## สรุปคะแนน
นับว่าผ่านกี่ข้อ แล้วบอก TA  ข้อที่ติดให้เปิดเฉลยด้านล่าง เทียบกับที่เขียนเอง

---
<details>
<summary><b>เฉลย (คลิกเพื่อเปิด)</b></summary>

**ข้อ 1:** `count_color(img, 95, 125)`

**ข้อ 2:**
```python
n_orange, _ = count_color(img, 12, 22)
verdict = "OK" if (n_red == 5 and n_orange == 0) else "NG"
```

**ข้อ 3:** `n_chip = sum(50 < a <= 1000 for a in areas)` และ `verdict = "OK" if (n_big == 5 and n_chip == 0) else "NG"`

**ข้อ 4:** `return re.search(r"LOT\s*\d{6}\b", text) is not None`

**ข้อ 5:** `green_ratio = cv2.inRange(roi, (40, 80, 80), (80, 255, 255)).mean() / 255`

**ข้อ 6:** `COIN_MM = 25.0`

**ข้อ 7:**
```python
res = cv2.matchTemplate(g_img, template, cv2.TM_CCOEFF_NORMED)
_, _, _, (x, y) = cv2.minMaxLoc(res)
dx, dy = x - 150, y - 100
```

**ข้อ 8:** เพิ่ม `"extra"` ใน CLASSES และใน make_tray:
```python
if kind == "extra":
    cv2.circle(img, (int(160 + dx), int(115 + dy)), 26, (190, 190, 195), -1)
    cv2.circle(img, (int(160 + dx), int(115 + dy)), 8, (80, 82, 85), -1)
```
</details>